In [ ]:
"""
Pull versioned ("as-of") data from Delphi Epidata via epidatpy.

Key lessons baked in:
- V5 renamed many signals (e.g. NHSN dropped the `_prelim` suffix), so the V5
  and V4 signal names can differ.
- V5 `report_time` is the actual publication date (Wed/Fri), NOT the epiweek
  Saturday. If your lookup keys on the epiweek, normalize it (Week.enddate()).
- Forecasts are made on the release day (Wed), so to get "data available at
  forecast time" take the EARLIEST report within the forecast week, not a later
  same-week revision.
- Delphi V5 rolls out source-by-source; if a source isn't on V5 yet, fall back
  to the V4 endpoint (still via epidatpy) until the V4 sunset (~Oct 2026).

Requires: pip install epidatpy pandas epiweeks   (epidatpy needs pandas>=2)
Auth: set env var DELPHI_EPIDATA_KEY (get a free key from Delphi).
"""
import os
from datetime import datetime, timedelta

import pandas as pd
from epiweeks import Week
from epidatpy import EpiDataContext, EpiRange


def fetch_versioned(
    source: str,
    signal_v5: str,
    signal_v4: str,
    geo_types=("state", "nation"),
    geo_values="*",
    lookback_days: int = 400,
    as_of: str = "wednesday",   # "wednesday" = earliest report in week; "latest" = newest
):
    """Return versioned rows: geo_value, value, reference_date (week-ending
    Saturday), issue_date (epiweek-ending Saturday of the report). Tries the V5
    `epidata_archive` endpoint first, then the V4 `pub_covidcast` endpoint."""
    end = datetime.now().date()
    start = end - timedelta(days=lookback_days)
    ctx = EpiDataContext(use_cache=False)

    # ---- V5 archive (preferred) ----
    frames = []
    for geo in geo_types:
        try:
            df = ctx.epidata_archive(
                source=source, signals=signal_v5,
                geo_type=geo, geo_values=geo_values,
                reference_time=EpiRange(start.isoformat(), end.isoformat()),
                report_time=EpiRange(start.isoformat(), end.isoformat()),
            ).df()
            if df is not None and len(df):
                frames.append(_normalize_v5(df, as_of))
        except Exception as e:
            print(f"  V5 archive failed ({geo}): {type(e).__name__}: {e}")
    if frames:
        return pd.concat(frames, ignore_index=True)

    # ---- V4 covidcast fallback (still via epidatpy) ----
    sw, ew = Week.fromdate(start), Week.fromdate(end)
    se, ee = int(f"{sw.year}{sw.week:02d}"), int(f"{ew.year}{ew.week:02d}")
    frames = []
    for geo in geo_types:
        try:
            df = ctx.pub_covidcast(
                data_source=source, signals=signal_v4, time_type="week",
                time_values=EpiRange(se, ee), geo_type=geo,
                geo_values=geo_values, issues=EpiRange(se, ee),
            ).df()
            if df is not None and len(df):
                frames.append(_normalize_v4(df, as_of))
        except Exception as e:
            print(f"  V4 covidcast failed ({geo}): {type(e).__name__}: {e}")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=["geo_value", "value", "reference_date", "issue_date"])


def _normalize_v5(df, as_of):
    d = df.rename(columns={"reference_time": "reference_date"}).copy()
    d["reference_date"] = pd.to_datetime(d["reference_date"]).apply(lambda x: Week.fromdate(x).enddate())
    raw = pd.to_datetime(d["report_time"])
    d["issue_date"] = raw.apply(lambda x: Week.fromdate(x).enddate())   # epiweek Saturday
    d["_raw"] = raw
    keep = "first" if as_of == "wednesday" else "last"                  # Wed prelim vs latest revision
    d = d.sort_values("_raw").drop_duplicates(
        subset=["geo_value", "reference_date", "issue_date"], keep=keep)
    return d[["geo_value", "value", "reference_date", "issue_date"]]


def _normalize_v4(df, as_of):
    d = df[["geo_value", "time_value", "issue", "value"]].copy()
    d["time_value"] = d["time_value"].apply(lambda x: int(str(x)))
    d["issue"] = d["issue"].apply(lambda x: int(str(x)))
    d["reference_date"] = d["time_value"].apply(lambda x: Week(x // 100, x % 100).enddate())
    d["issue_date"] = d["issue"].apply(lambda x: Week(x // 100, x % 100).enddate())
    # V4 issue is already epiweek-granular (one per week), so as_of mostly N/A
    return d[["geo_value", "value", "reference_date", "issue_date"]]


if __name__ == "__main__":
    os.environ.setdefault("DELPHI_EPIDATA_KEY", os.environ.get("COVIDCAST_API_KEY", ""))
    out = fetch_versioned(
        source="nhsn",
        signal_v5="confirmed_admissions_flu_ew",
        signal_v4="confirmed_admissions_flu_ew_prelim",
    )
    print(out.head(), "\n", len(out), "rows")